In [ ]:
"""Apply trained recommender models to players from the current season."""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from ml.toolkit.ml_utilities import (
    POSITION_GROUPS,
    align_prediction_features,
    build_recommender_season_features,
    get_recommender_feature_columns,
    load_model,
)


DATA_DIR = PROJECT_ROOT / "data" / "transform"
PLAYER_STATS_PATH = DATA_DIR / "player_stats.csv"
MATCHES_PATH = DATA_DIR / "matches.csv"
PLAYERS_PATH = DATA_DIR / "players.csv"
MODEL_PATH_TEMPLATE = CURRENT_DIR / "recommender_model_{position_group}.pkl"
PREDICTION_SEASON = 2025
PREDICTION_COLUMN = "prediction"
PLAYER_ID_COLUMN = "player_id"
POSITION_GROUP_COLUMN = "position_group"


def load_prediction_tables() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load transformed tables required for recommender inference."""
    required_files = [PLAYER_STATS_PATH, MATCHES_PATH, PLAYERS_PATH]
    missing_files = [str(path) for path in required_files if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing prediction files: {missing_files}")

    player_stats = pd.read_csv(PLAYER_STATS_PATH)
    matches = pd.read_csv(MATCHES_PATH)
    players = pd.read_csv(PLAYERS_PATH)

    return player_stats, matches, players


def load_position_group_model(position_group: str):
    """Load a recommender model for one position group if it exists."""
    model_path = Path(str(MODEL_PATH_TEMPLATE).format(position_group=position_group))
    if not model_path.exists():
        print(f"Skipped {position_group}: model file not found at {model_path}")
        return None
    return load_model(model_path)


def predict_position_group(
    prediction_dataset: pd.DataFrame,
    position_group: str,
) -> pd.DataFrame:
    """Predict next-season ratings for one position group."""
    model = load_position_group_model(position_group)
    if model is None:
        return pd.DataFrame(columns=[PLAYER_ID_COLUMN, PREDICTION_COLUMN])

    group_dataset = prediction_dataset[
        prediction_dataset[POSITION_GROUP_COLUMN] == position_group
    ].copy()

    if group_dataset.empty:
        return pd.DataFrame(columns=[PLAYER_ID_COLUMN, PREDICTION_COLUMN])

    feature_columns = getattr(model, "feature_columns_", get_recommender_feature_columns(group_dataset))
    features = align_prediction_features(group_dataset, feature_columns)

    group_predictions = group_dataset[[PLAYER_ID_COLUMN]].copy()
    group_predictions[PREDICTION_COLUMN] = model.predict(features).round(2)

    return group_predictions


def predict_current_season_players(
    player_stats: pd.DataFrame,
    matches: pd.DataFrame,
    players: pd.DataFrame,
) -> pd.DataFrame:
    """Predict next-season ratings for players who played in the configured season."""
    season_features = build_recommender_season_features(player_stats, matches, players)
    prediction_dataset = season_features[season_features["season"] == PREDICTION_SEASON].copy()
    prediction_dataset = prediction_dataset[prediction_dataset[POSITION_GROUP_COLUMN].notna()].copy()

    prediction_frames = []
    for position_group in POSITION_GROUPS:
        prediction_frames.append(predict_position_group(prediction_dataset, position_group))

    if not prediction_frames:
        return pd.DataFrame(columns=[PLAYER_ID_COLUMN, PREDICTION_COLUMN])

    predictions = pd.concat(prediction_frames, ignore_index=True)
    predictions = predictions.dropna(subset=[PREDICTION_COLUMN])
    return predictions.drop_duplicates(subset=[PLAYER_ID_COLUMN], keep="last")


def update_players_with_predictions(
    players: pd.DataFrame,
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    """Add predictions to the players table while keeping non-playing players as missing values."""
    result = players.drop(columns=[PREDICTION_COLUMN], errors="ignore").copy()
    result = result.merge(predictions, on=PLAYER_ID_COLUMN, how="left")
    result[PREDICTION_COLUMN] = pd.to_numeric(result[PREDICTION_COLUMN], errors="coerce")
    return result


def print_top_predictions(players: pd.DataFrame) -> None:
    """Print the five players with the highest predictions."""
    name_columns = [column for column in ["player_name", "name"] if column in players.columns]
    display_columns = [PLAYER_ID_COLUMN] + name_columns + [PREDICTION_COLUMN]
    top_players = players.dropna(subset=[PREDICTION_COLUMN]).sort_values(PREDICTION_COLUMN, ascending=False)
    print("\nTop 5 predicted players:")
    print(top_players[display_columns].head(5).to_string(index=False))


def save_players(players: pd.DataFrame) -> None:
    """Save the players table with the prediction column."""
    players.to_csv(PLAYERS_PATH, index=False)
    predicted_count = players[PREDICTION_COLUMN].notna().sum()
    print(f"Saved players with predictions: {PLAYERS_PATH}")
    print(f"Predicted players: {predicted_count}")


def main() -> None:
    """Run recommender inference for the current season."""
    player_stats, matches, players = load_prediction_tables()
    predictions = predict_current_season_players(player_stats, matches, players)
    updated_players = update_players_with_predictions(players, predictions)
    save_players(updated_players)
    print_top_predictions(updated_players)


if __name__ == "__main__":
    main()
